# Fase 1 — Escolha da janela de anos (2023 a 2025)

Este notebook responde uma pergunta só: **dá para usar 2023, 2024 e 2025
como os três anos da base?** Ele também gera a **Figura 1** do artigo.

**O risco que estamos testando.** Os dados de crime vêm da SSP-SP
(Secretaria da Segurança Pública do Estado de São Paulo). Entre 2022 e 2023
a SSP trocou o sistema onde os boletins de ocorrência são registrados: saiu
o R.D.O. (Registro Digital de Ocorrências) e entrou o S.P.J. (Sistema de
Polícia Judiciária). Uma troca dessas muda o jeito de registrar, não a
quantidade de crime que acontece. Se os dois sistemas contam de formas
diferentes, juntar 2022 com 2023 faria a gente ler como "mudança na
criminalidade" uma variação que é só do software.

**A regra, combinada antes de olhar o gráfico.** Definir o critério antes
evita escolher depois a conclusão mais conveniente:

- se a série mensal der um **salto** entre dez/2022 e jan/2023, a troca de
  sistema ainda estava mexendo nos números e a janela passa a ser
  2024–2025;
- se a série passar **sem salto**, a janela 2023–2025 fica confirmada.

**Os dados.** Vêm de `data/processed/ssp_painel.csv`, que o `main.py` já
gerou com as ocorrências somadas por município, ano e mês. Os arquivos
originais da SSP têm milhões de linhas e levam minutos para ler, por isso
não são abertos de novo aqui.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Os notebooks ficam em notebooks/ e o código do projeto em src/. Estas duas
# linhas apontam o Python para src/, para os "from config import ..." abaixo
# funcionarem tanto rodando daqui quanto da raiz do projeto.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

from config import SSP_PAINEL_CSV, GRUPOS_NATUREZA
from estilo import aplicar_estilo, salvar
from figuras import plot_serie_mensal
from parse_ssp import normaliza

aplicar_estilo()   # mesma fonte, cores e grade em todas as figuras do artigo
# Se algum import acima falhar, quase sempre é o editor apontando para outra
# instalação do Python. O caminho impresso aqui é o que precisa ter as
# bibliotecas do requirements.txt.
print("Python:", sys.executable)

# O código do IBGE é identificador, não quantidade: lido como texto para não
# virar número em nenhuma etapa.
painel = pd.read_csv(SSP_PAINEL_CSV, dtype={"codigo_ibge": str})
print(painel.shape, "| anos:", sorted(painel["ano"].unique()))

## 1. A série está completa?

Antes de procurar um salto no gráfico, é preciso garantir que ele não seria
só um mês faltando. Um mês ausente vira um buraco na linha e se parece com
uma queda real.

São duas conferências. A primeira é quantas ocorrências vieram sem a data do
mês preenchida: elas ficam com `mes = 0` no painel, continuam contando no
total do ano e só não têm onde ser colocadas na linha do tempo. A segunda é
se cada ano tem os seus 12 meses.

In [ ]:
sem_mes = painel.loc[painel["mes"] == 0, "ocorrencias"].sum()
print(f"ocorrências sem mês: {sem_mes} de {painel['ocorrencias'].sum():,}")

# nunique() conta quantos meses diferentes cada ano tem. O esperado é 12.
meses_por_ano = painel[painel["mes"] > 0].groupby("ano")["mes"].nunique()
print(meses_por_ano.to_string())
# O assert derruba o notebook se faltar mês, em vez de deixar a gente
# interpretar um gráfico furado como se fosse um resultado.
assert (meses_por_ano == 12).all(), "algum ano não tem os 12 meses"

## 2. Figura 1 — total de ocorrências mês a mês, de 2022 a 2025

A linha vertical marca janeiro de 2023, o começo da janela. É ali que o
salto apareceria, se existisse.

O eixo vertical começa em zero de propósito. Quando se corta o eixo, uma
oscilação normal de poucos por cento passa a ocupar metade da altura do
gráfico e vira um degrau que não existe.

In [ ]:
fig = plot_serie_mensal(painel)
salvar(fig, "figura1_serie_mensal")

## 3. A mesma série, agora por tipo de crime

O total pode esconder movimentos contrários: uma queda no furto compensada
por uma alta no roubo dá uma linha total lisa. Por isso repetimos o gráfico
separando os dez grupos de crime que o projeto usa ("natureza" é o nome que
a SSP dá ao tipo de ocorrência).

Cada painel tem a sua própria escala, porque as quantidades são muito
diferentes entre si. O que interessa aqui é o **formato** de cada linha na
virada do ano, não a altura dela.

In [ ]:
fig = plot_serie_mensal(painel, GRUPOS_NATUREZA)
salvar(fig, "figura1b_serie_por_natureza")

## 4. Conferindo com números, e não só com o olho

O gráfico é sugestivo, mas duas coisas enganam a vista. A primeira é a
sazonalidade: crime tem época do ano, então toda passagem de dezembro para
janeiro já muda de nível sozinha. A segunda é que um salto pequeno,
espalhado em dez painéis, é difícil de enxergar.

O teste então é comparativo. Calculamos, para cada tipo de crime, a variação
percentual de um ano para o outro, e perguntamos duas coisas:

1. a variação de 2022 para 2023 é **maior** que a das outras viradas de ano?
2. ela vai **na mesma direção** em todos os tipos de crime?

Uma troca de sistema de registro faria as duas coisas ao mesmo tempo, porque
atingiria todos os crimes de uma vez. Sazonalidade e variação normal, não.

In [ ]:
# GRUPOS_NATUREZA (config.py) diz quais rubricas da SSP formam cada grupo.
# Aqui o dicionário é invertido para "rubrica -> grupo". normaliza() tira
# acento e caixa, porque o texto da planilha muda de um ano para o outro.
mapa = {normaliza(nat): g for g, lista in GRUPOS_NATUREZA.items() for nat in lista}
# O dropna descarta as rubricas que o projeto não usa (trânsito, porte de
# arma), que ficaram sem grupo no map acima.
bloco = painel.assign(grupo=painel["natureza"].map(mapa)).dropna(subset=["grupo"])

# unstack("ano") joga os anos para as colunas: uma linha por grupo de crime,
# uma coluna por ano.
anual = bloco.groupby(["grupo", "ano"])["ocorrencias"].sum().unstack("ano")
# pct_change(axis=1) compara cada ano com o da coluna anterior. A primeira
# coluna sai (iloc[:, 1:]) porque não tem ano anterior com que comparar.
variacao = (anual.pct_change(axis=1) * 100).round(1).iloc[:, 1:]
variacao.columns = [f"{a - 1}→{a}" for a in anual.columns[1:]]
variacao

In [ ]:
primeira = variacao.iloc[:, 0]          # a virada suspeita: 2022-2023
demais = variacao.iloc[:, 1:]           # 2023-2024 e 2024-2025: o padrão normal

# Pergunta 2: se fosse troca de sistema, quase todos os crimes iriam para o
# mesmo lado.
print(f"2022-2023: {(primeira > 0).sum()} naturezas sobem, "
      f"{(primeira < 0).sum()} descem")
# Pergunta 1: o tamanho da variação, sem o sinal (abs), comparado com o das
# viradas normais. O stack() junta as duas colunas de "demais" numa lista só,
# para tirar uma mediana de todas elas de uma vez.
print(f"variação mediana (em módulo): 2022-2023 = {primeira.abs().median():.1f}% | "
      f"demais viradas = {demais.abs().stack().median():.1f}%")

# Critério de "fora do padrão": variar mais que o dobro do que aquele mesmo
# crime costuma variar nas outras viradas de ano.
fora = primeira.abs() > 2 * demais.abs().median(axis=1)
print("naturezas em que 2022-2023 passa do dobro do normal:",
      ", ".join(fora[fora].index) or "nenhuma")

## 5. Conclusão: janela 2023–2025 confirmada

Os quatro resultados apontam para o mesmo lado.

- A linha do total não tem salto em janeiro de 2023.
- Na virada de 2022 para 2023, metade dos tipos de crime sobe e metade
  desce. Uma troca de sistema de registro empurraria quase todos juntos para
  o mesmo lado, e não foi o que aconteceu.
- A variação mediana dessa virada é praticamente igual à das outras viradas
  de ano, ou seja, 2022-2023 não tem nada de excepcional.
- Os dois únicos tipos fora do padrão são o CVLI (crimes violentos letais
  intencionais: homicídio doloso, latrocínio e lesão corporal seguida de
  morte) e o estupro. São justamente os que menos dependem do sistema de
  registro, porque são os de notificação mais obrigatória e menos sujeitos a
  mudança de classificação. Uma troca de software apareceria primeiro no
  furto e no roubo, que são os de maior volume e registro mais burocrático.

Por isso `ANOS_JANELA` continua `[2023, 2024, 2025]` no `config.py`. O ano de
2022 fica no painel intermediário só para servir de comparação e não entra
na base final.